# 第三章：搭建多模态桥梁 — VLM 架构

> 上一章，CLIP 让图像和文本拥有了「共同语言」——它们的嵌入向量处于同一空间。
> 但这还不够：CLIP 只能告诉你「图像和文字是否匹配」，
> 而不能「看着图像生成文字」。
> 本章解决这个问题，构建第一个能回答图像问题的 VLM。

---

## 本章目标

从零实现 **LLaVA 风格的 Vision-Language Model**，理解：

1. 图像与文本特征空间的「语义鸿沟」
2. **Projection MLP**：如何用极少参数桥接两个大模型
3. **Visual Prefix**：图像 token 如何插入语言模型的输入序列
4. 训练时的损失掩码：为什么只在文本 token 上计算损失
5. 完整 VLM 的前向传播，逐步验证每个形状变换

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), '.'))
os.makedirs('figures', exist_ok=True)

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

---
## 3.1 为什么需要 Projection 层？

### 3.1.1 两个空间的语义鸿沟

我们有两个预训练模型：

- **ViT（图像编码器）**：在数亿张图文对上做 CLIP 训练，输出 768 维向量
- **GPT（语言模型）**：在数万亿 token 的文本上预训练，接受 768 维 token 嵌入

两者都是 768 维，能不能直接把 ViT 的输出当作 GPT 的 token 输入？

**答案：不行。**

虽然维度相同，但两个空间的含义完全不同：

| | ViT 特征空间 | GPT 嵌入空间 |
|--|--|--|
| 训练目标 | 图文对比（余弦相似度） | 下一词预测（cross-entropy） |
| 范数分布 | L2 归一化（长度=1） | 无约束（长度可以很大） |
| 语义组织 | 视觉语义聚类 | 语言统计关联 |
| 典型值范围 | [-1, 1]（归一化后） | [-2, 2]（训练后经验值） |

直接输入会破坏 GPT 的内部表示，导致模型困惑甚至崩溃。

### 3.1.2 Projection MLP 的作用

我们需要一个**可学习的转换器**，把 ViT 空间里的向量翻译成 GPT 空间里有意义的 token。

LLaVA-1.5 用的是一个 **两层 MLP**：

```
Linear(vision_dim → language_dim) → GELU → Linear(language_dim → language_dim)
```

为什么不用单层线性变换（更简单）？
- 线性变换只能做旋转/缩放，表达能力有限
- GELU 激活函数引入非线性，可以学到更复杂的映射
- 实验表明，两层 MLP 比单层线性提升约 2-3 个点（VQA 准确率）

In [ ]:
# 演示特征空间差异
vision_dim   = 768
language_dim = 768

# 模拟 ViT 输出（L2 归一化，所有向量在单位球面上）
vit_features = F.normalize(torch.randn(196, vision_dim), dim=-1)

# 模拟 GPT token 嵌入（来自词表，无归一化约束）
gpt_embeddings = torch.randn(196, language_dim) * 0.5  # 训练后分布

print("=== 两个特征空间的统计差异 ===")
print(f"ViT 特征 — 均值范数: {vit_features.norm(dim=-1).mean():.3f}  "
      f"标准差: {vit_features.norm(dim=-1).std():.4f}")
print(f"GPT 嵌入 — 均值范数: {gpt_embeddings.norm(dim=-1).mean():.3f}  "
      f"标准差: {gpt_embeddings.norm(dim=-1).std():.4f}")

print("\n如果直接把 ViT 输出插入 GPT：")
print(f"  ViT 向量范数: ~1.0  vs  GPT 期望范数: ~{gpt_embeddings.norm(dim=-1).mean():.1f}")
print(f"  这相当于给 GPT 输入了非常'轻'的 token，模型会困惑")

In [ ]:
class ProjectionMLP(nn.Module):
    """
    两层 MLP，将视觉特征投影到语言模型的嵌入空间。

    这是整个 VLM 中参数量最少的组件（通常不到总参数的 1%），
    却承担着最关键的「翻译」工作。

    设计选择：
    - 用 GELU 而非 ReLU：保留负数区域的梯度，训练更稳定
    - 不用 LayerNorm：避免破坏投影后向量的统计特性
    - 输出无激活：保持输出范围与 GPT token 嵌入一致
    """

    def __init__(self, vision_dim: int, language_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(vision_dim, language_dim),
            nn.GELU(),
            nn.Linear(language_dim, language_dim),
        )
        # 初始化：输出接近 0，避免在训练初期严重干扰 GPT
        nn.init.normal_(self.net[0].weight, std=0.02)
        nn.init.zeros_(self.net[0].bias)
        nn.init.normal_(self.net[2].weight, std=0.02)
        nn.init.zeros_(self.net[2].bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, n_img_tokens, vision_dim)
        # 返回: (B, n_img_tokens, language_dim)
        return self.net(x)


# 验证
proj = ProjectionMLP(vision_dim=768, language_dim=768)
vit_out = torch.randn(2, 196, 768)   # (B, n_patches, D_vision)
proj_out = proj(vit_out)             # (B, n_patches, D_language)

n_params = sum(p.numel() for p in proj.parameters())
print(f"Projection MLP 参数量: {n_params:,}")
print(f"= 768×768 + 768 + 768×768 + 768 = {768*768+768+768*768+768:,}")
print(f"\n输入 (ViT 特征):  {vit_out.shape}")
print(f"输出 (GPT token): {proj_out.shape}")
print(f"\n投影后输出均值范数: {proj_out.norm(dim=-1).mean().item():.3f}")
print(f"（接近 GPT token 嵌入的范数分布）")

---
## 3.2 Visual Prefix：让语言模型「看见」图像

### 3.2.1 核心思想

GPT 本质上是一个**序列处理器**：给定一段 token 序列，预测下一个 token。

我们的想法：**把图像的视觉 token 插入到文本 token 之前**，作为一种「视觉上下文」。

```
原始 GPT 输入:   [USER: 这是什么颜色？ ASSISTANT: ]
加入视觉后:      [V1, V2, ..., V196, USER: 这是什么颜色？ ASSISTANT: ]
                 ↑                    ↑
              196 个视觉 token      原始文本 token
```

GPT 的因果注意力会让文本 token 能「回望」前面的视觉 token，
从而在生成时参考图像信息。

### 3.2.2 训练时的损失掩码

一个容易忽略的细节：

**我们只在文本 token 上计算损失，视觉 token 的损失被掩码为 -100（忽略）。**

原因：
- 视觉 token 不是词表中的合法 token，没有「下一个词」的概念
- 对视觉 token 计算 cross-entropy 会引入无意义的梯度，干扰训练
- 只需要语言模型学会「基于视觉上下文生成文字」

In [ ]:
# 演示 Visual Prefix 的拼接过程和损失掩码
B = 2
n_img_tokens = 4   # 简化演示用 4 个
T = 6              # 文本长度
D = 8              # 嵌入维度（示例用）
vocab_size = 100

# 1. 图像经过 ViT + Projection，得到视觉 token
visual_tokens = torch.randn(B, n_img_tokens, D)   # (B, 4, D)

# 2. 文本 token 序列
text_ids = torch.randint(0, vocab_size, (B, T))    # (B, 6)
token_emb = nn.Embedding(vocab_size, D)
text_tokens = token_emb(text_ids)                  # (B, 6, D)

# 3. 拼接：视觉前缀 + 文本
full_seq = torch.cat([visual_tokens, text_tokens], dim=1)  # (B, 10, D)

# 4. 损失标签：视觉部分填 -100（忽略），文本部分用原始 token id
labels = torch.cat([
    torch.full((B, n_img_tokens), -100),  # 视觉 token → 忽略
    text_ids,                              # 文本 token → 计算损失
], dim=1)

print("=== Visual Prefix 拼接示意 ===")
print(f"视觉 token:   {visual_tokens.shape}   (B, n_img, D)")
print(f"文本 token:   {text_tokens.shape}   (B, T, D)")
print(f"完整序列:     {full_seq.shape}  (B, n_img+T, D)")
print(f"\n损失标签 (第1个样本): {labels[0].tolist()}")
print(f"  -100 = 忽略（视觉 token），其余 = 目标 token id")
print(f"\n实际计算损失的位置数: {(labels != -100).sum().item()} / {labels.numel()}")

In [ ]:
from multimodal_from_scratch.figures import draw_vlm_architecture
fig = draw_vlm_architecture(save_path='figures/ch03_vlm_arch.png')
plt.show()

---
## 3.3 GPT 解码器回顾

如果你读完了原书，GPT 的实现你已经非常熟悉。
这里我们实现一个精简版，专注于 **`visual_prefix` 接口**——这是 VLM 扩展 GPT 的关键钩子。

In [ ]:
class CausalSelfAttention(nn.Module):
    """因果自注意力（原书已详解，这里只保留核心代码）"""

    def __init__(self, embed_dim, num_heads, context_len, dropout=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.qkv       = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.out_proj  = nn.Linear(embed_dim, embed_dim, bias=False)
        self.drop      = nn.Dropout(dropout)
        # 因果掩码：上三角 = True（被阻止的位置）
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_len, context_len), diagonal=1).bool()
        )

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.masked_fill(self.mask[:T, :T], float('-inf'))  # 因果掩码
        attn = F.softmax(attn, dim=-1)
        out  = (self.drop(attn) @ v).transpose(1, 2).reshape(B, T, C)
        return self.out_proj(out)


class GPTBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, context_len, ffn_mult=4, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = CausalSelfAttention(embed_dim, num_heads, context_len, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        h = embed_dim * ffn_mult
        self.ffn   = nn.Sequential(
            nn.Linear(embed_dim, h), nn.GELU(), nn.Linear(h, embed_dim), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class GPTDecoder(nn.Module):
    """
    GPT 语言模型解码器。

    与原书相比，唯一新增的是 `visual_prefix` 参数：
    如果提供，会在 token 嵌入前面拼接视觉 token，
    让语言模型能在生成时参考图像信息。
    """

    def __init__(self, vocab_size, context_len, embed_dim, depth, num_heads, dropout=0.0):
        super().__init__()
        self.context_len = context_len
        self.embed_dim   = embed_dim

        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed   = nn.Embedding(context_len, embed_dim)
        self.drop        = nn.Dropout(dropout)
        self.blocks      = nn.ModuleList([
            GPTBlock(embed_dim, num_heads, context_len, dropout=dropout)
            for _ in range(depth)
        ])
        self.norm    = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)
        # 权重绑定（详见原书 Ch4）
        self.lm_head.weight = self.token_embed.weight

    def forward(self, input_ids, visual_prefix=None):
        """
        Args:
            input_ids:     (B, T)           文本 token 序列
            visual_prefix: (B, N_img, D)    可选，视觉 token（来自 ViT + Projection）
        Returns:
            logits: (B, N_img + T, vocab_size)
        """
        B, T = input_ids.shape
        pos  = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x    = self.token_embed(input_ids) + self.pos_embed(pos)   # (B, T, D)

        # 关键：如果有视觉前缀，拼接到序列头部
        if visual_prefix is not None:
            x = torch.cat([visual_prefix, x], dim=1)  # (B, N_img + T, D)

        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.lm_head(x)

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0,
                 top_k=None, visual_prefix=None):
        for step in range(max_new_tokens):
            ids_cond = input_ids[:, -self.context_len:]
            vp = visual_prefix if step == 0 else None
            logits = self(ids_cond, visual_prefix=vp)[:, -1, :] / temperature
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')
            next_id = torch.multinomial(F.softmax(logits, dim=-1), 1)
            input_ids = torch.cat([input_ids, next_id], dim=1)
        return input_ids


# 验证 visual_prefix 接口
gpt = GPTDecoder(vocab_size=1000, context_len=128, embed_dim=128, depth=2, num_heads=4)

input_ids = torch.randint(0, 1000, (2, 10))          # 文本
vis_prefix = torch.randn(2, 16, 128)                  # 视觉前缀（16 个 token）

# 不带视觉
logits_text_only = gpt(input_ids)
# 带视觉前缀
logits_with_vis  = gpt(input_ids, visual_prefix=vis_prefix)

print(f"仅文本输入:     logits shape = {logits_text_only.shape}")
print(f"有视觉前缀:     logits shape = {logits_with_vis.shape}")
print(f"  = (B=2, 16个视觉token + 10个文本token, vocab=1000)")

---
## 3.4 组装完整 VLM

现在把三个组件拼在一起：
1. **ViT**（图像编码器，第1章）
2. **ProjectionMLP**（桥接层，本章 3.1）
3. **GPT**（语言解码器，原书 + visual_prefix 扩展）

In [ ]:
# 复用第1章的 ViT 组件
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
    def forward(self, x): return self.proj(x).flatten(2).transpose(1, 2)

class PositionalEmbedding(nn.Module):
    def __init__(self, n, d):
        super().__init__()
        self.pe = nn.Parameter(torch.zeros(1, n+1, d))
    def forward(self, x): return x + self.pe

class MHSA_Bi(nn.Module):   # 双向（ViT 用）
    def __init__(self, d, h):
        super().__init__()
        self.h=h; self.hd=d//h; self.s=self.hd**-0.5
        self.qkv=nn.Linear(d,3*d); self.o=nn.Linear(d,d)
    def forward(self,x):
        B,N,C=x.shape
        q,k,v=self.qkv(x).reshape(B,N,3,self.h,self.hd).permute(2,0,3,1,4).unbind(0)
        return self.o((F.softmax((q@k.transpose(-2,-1))*self.s,dim=-1)@v).transpose(1,2).reshape(B,N,C))

class ViTBlockMini(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.n1=nn.LayerNorm(d); self.a=MHSA_Bi(d,h)
        self.n2=nn.LayerNorm(d); self.f=nn.Sequential(nn.Linear(d,d*4),nn.GELU(),nn.Linear(d*4,d))
    def forward(self,x): x=x+self.a(self.n1(x)); return x+self.f(self.n2(x))

class ViTEncoder(nn.Module):
    def __init__(self, img_size, patch_size, embed_dim, depth, num_heads):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, 3, embed_dim)
        n = self.patch_embed.n_patches
        self.cls = nn.Parameter(torch.zeros(1,1,embed_dim))
        self.pos = PositionalEmbedding(n, embed_dim)
        self.blocks = nn.Sequential(*[ViTBlockMini(embed_dim, num_heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        x = torch.cat([self.cls.expand(B,-1,-1), x], dim=1)
        x = self.norm(self.blocks(self.pos(x)))
        return x[:, 1:]   # 返回 patch tokens（去掉 CLS），保留空间信息

print("ViT 和 GPT 组件已定义 ✓")

In [ ]:
class VisionLanguageModel(nn.Module):
    """
    LLaVA 风格的 Vision-Language Model。

    组件：
      - ViTEncoder:     图像 → patch tokens  (B, N_img, D_v)
      - ProjectionMLP:  (B, N_img, D_v) → (B, N_img, D_l)  [视觉→语言空间]
      - GPTDecoder:     [视觉前缀 | 文本] → logits

    训练策略（详见第4章）：
      Stage 1: 只训练 ProjectionMLP，ViT 和 GPT 冻结
      Stage 2: 训练 ProjectionMLP + GPT，ViT 冻结
    """

    def __init__(self, img_size=32, patch_size=8,
                 vision_dim=64, vision_depth=2, vision_heads=4,
                 vocab_size=500, context_len=64,
                 language_dim=64, language_depth=2, language_heads=4):
        super().__init__()

        # 1. 图像编码器
        self.vision_encoder = ViTEncoder(
            img_size, patch_size, vision_dim, vision_depth, vision_heads)

        # 2. 投影层（最小但最关键的组件）
        self.projection = ProjectionMLP(
            vision_dim=vision_dim, language_dim=language_dim)

        # 3. 语言解码器
        self.language_model = GPTDecoder(
            vocab_size, context_len, language_dim, language_depth, language_heads)

    # ── 训练阶段控制 ───────────────────────────────────────────────

    def freeze_vision(self):
        for p in self.vision_encoder.parameters():
            p.requires_grad = False

    def freeze_language(self):
        for p in self.language_model.parameters():
            p.requires_grad = False

    def unfreeze_language(self):
        for p in self.language_model.parameters():
            p.requires_grad = True

    def set_stage1(self):
        """Stage 1：特征对齐，只训练 Projection MLP"""
        self.freeze_vision()
        self.freeze_language()
        for p in self.projection.parameters():
            p.requires_grad = True

    def set_stage2(self):
        """Stage 2：指令微调，训练 Projection + LLM"""
        self.freeze_vision()
        self.unfreeze_language()
        for p in self.projection.parameters():
            p.requires_grad = True

    # ── 核心前向传播 ───────────────────────────────────────────────

    def encode_image(self, images):
        """图像 → 视觉 token（语言空间）"""
        patch_feats    = self.vision_encoder(images)   # (B, N_img, D_v)
        visual_tokens  = self.projection(patch_feats)  # (B, N_img, D_l)
        return visual_tokens

    def forward(self, images, input_ids, labels=None):
        """
        Args:
            images:    (B, 3, H, W)
            input_ids: (B, T)    文本 token
            labels:    (B, T)    损失目标；-100 = 忽略（视觉对应位置）
        Returns:
            dict: 'logits', 可选 'loss'
        """
        visual_tokens = self.encode_image(images)                    # (B, N_img, D_l)
        logits = self.language_model(input_ids, visual_prefix=visual_tokens)  # (B, N_img+T, V)

        result = {'logits': logits}

        if labels is not None:
            n_img = visual_tokens.shape[1]
            # 损失只计算文本部分（去掉视觉 token 对应的 logit，并左移一位做 next-token）
            text_logits = logits[:, n_img:-1, :]   # (B, T-1, V)
            text_labels = labels[:, 1:]            # (B, T-1)
            result['loss'] = F.cross_entropy(
                text_logits.reshape(-1, text_logits.shape[-1]),
                text_labels.reshape(-1),
                ignore_index=-100
            )

        return result

    @torch.no_grad()
    def generate(self, images, prompt_ids, max_new_tokens=50, temperature=1.0, top_k=50):
        """给定图像和文字提示，自回归生成回答"""
        visual_tokens = self.encode_image(images)
        return self.language_model.generate(
            prompt_ids, max_new_tokens=max_new_tokens,
            temperature=temperature, top_k=top_k,
            visual_prefix=visual_tokens
        )

    def count_params(self, trainable_only=False):
        fn = (lambda p: p.requires_grad) if trainable_only else (lambda p: True)
        return {
            'vision_encoder': sum(p.numel() for p in self.vision_encoder.parameters() if fn(p)),
            'projection':     sum(p.numel() for p in self.projection.parameters() if fn(p)),
            'language_model': sum(p.numel() for p in self.language_model.parameters() if fn(p)),
            'total':          sum(p.numel() for p in self.parameters() if fn(p)),
        }

In [ ]:
# 构建并检查参数分布
vlm = VisionLanguageModel(
    img_size=32, patch_size=8,
    vision_dim=128, vision_depth=3, vision_heads=4,
    vocab_size=500, context_len=64,
    language_dim=128, language_depth=3, language_heads=4,
)

params = vlm.count_params(trainable_only=False)
total  = params['total']

print("=== VLM 参数分布 ===")
for k, v in params.items():
    if k != 'total':
        print(f"  {k:<20}: {v:>8,}  ({v/total*100:5.1f}%)")
print(f"  {'总计':<20}: {total:>8,}")

print("\n注意：Projection MLP 只占极小比例，")
print("但它是连接图像和文字的唯一可学习桥梁。")

In [ ]:
# 逐步验证完整前向传播的形状变换
B = 2
images    = torch.randn(B, 3, 32, 32)
input_ids = torch.randint(0, 500, (B, 10))
labels    = input_ids.clone()
labels[:, :3] = -100   # 前 3 个 token（模拟用户提问部分）不计损失

print("=== 前向传播各阶段形状 ===")
print(f"输入图像:         {images.shape}")

with torch.no_grad():
    patch_feats = vlm.vision_encoder(images)
    print(f"  → ViT patch tokens:   {patch_feats.shape}   (B, N_patches, D_vision)")

    visual_tok = vlm.projection(patch_feats)
    print(f"  → Projection 输出:    {visual_tok.shape}   (B, N_patches, D_language)")

    text_emb = vlm.language_model.token_embed(input_ids)
    print(f"\n文本 token:           {text_emb.shape}")

    full_input = torch.cat([visual_tok, text_emb], dim=1)
    print(f"  → 拼接后完整序列:     {full_input.shape}   (B, N_patches+T, D_l)")

out = vlm(images, input_ids, labels)
n_img = vlm.vision_encoder.patch_embed.n_patches
print(f"\n最终 logits:          {out['logits'].shape}")
print(f"  = (B, {n_img} 视觉 + 10 文本, vocab=500)")
print(f"\n训练 loss:            {out['loss'].item():.4f}")
print(f"  （只对文本 token 计算，视觉 token 被 -100 忽略）")

---
## 3.5 各阶段参数量对比

Stage 1 只训练 Projection，来看看它占总参数的比例：

In [ ]:
vlm.set_stage1()
s1 = vlm.count_params(trainable_only=True)

vlm.set_stage2()
s2 = vlm.count_params(trainable_only=True)

all_params = vlm.count_params(trainable_only=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
components = ['vision_encoder', 'projection', 'language_model']
colors     = ['#5DADE2', '#F39C12', '#2ECC71']
names      = ['ViT Encoder', 'Projection MLP', 'GPT Decoder']

for ax, stage_params, title in [
    (axes[0], s1, 'Stage 1：仅训练 Projection MLP'),
    (axes[1], s2, 'Stage 2：训练 Projection + GPT'),
]:
    values = [stage_params[c] for c in components]
    total  = all_params['total']
    bars   = ax.bar(names, [v/total*100 for v in values], color=colors, edgecolor='white', lw=1.5)
    for bar, v in zip(bars, values):
        pct = v/total*100
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'{pct:.1f}%\n({v:,})', ha='center', va='bottom', fontsize=9)
    ax.set_ylabel('占总参数比例 (%)')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylim(0, 65)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('VLM 两阶段训练的参数规模', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/ch03_param_dist.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 本章小结

### 我们构建了什么

| 组件 | 关键点 |
|------|--------|
| `ProjectionMLP` | 两层 MLP 桥接视觉/语言空间；参数极少但作用关键 |
| `GPTDecoder` | 新增 `visual_prefix` 参数，兼容纯文本和图文混合输入 |
| `VisionLanguageModel` | 三组件组装；损失只算文本 token |
| 训练阶段控制 | `set_stage1()` / `set_stage2()` 精确控制冻结/解冻 |

### 核心洞察：为什么这个简单设计能 work？

- GPT 已经学会了「根据上下文生成文字」
- ViT 已经学会了「把图像编码成有意义的向量」
- 我们只需要教 Projection MLP：「如何把 ViT 的语言翻译成 GPT 能听懂的」
- 这个翻译问题比「从头学会看图说话」简单得多

### 下一章预告

架构有了，现在需要**训练**它。第4章详细实现两阶段训练，
并通过消融实验展示为什么要分两个阶段——跳过 Stage 1 会发生什么。